## This code explain the steps used in preparing our paper and results
Make sure to install the requirements by running the next line

In [ ]:
#pip install -r requirements.txt


# 1. Generating SPL Configurations Data Code 

In [ ]:
# Importing necessary libraries
import pandas as pd  # For data manipulation and DataFrames
import numpy as np   # For numerical operations and random number generation
import csv           # For CSV files

# Function to generate input configurations
def generate_input_configurations(number_of_p, number_of_r, p_maximum_range, l_r, h_r, number_of_class=1,
                                  number_of_rows_for_each_line=15000, max_buffer_range=50, range_of_machines=20):
    
    # Step 1: transition matrices generation
    matrices = []
    
    # 'number_of_p' random p-values are generatefrom a uniform distribution between 0.01 and p_maximum_range
    p_values = np.random.uniform(0.01, p_maximum_range, number_of_p)
    
    # For each p-value,  'number_of_r' r-values are generated based on scaled range [l_r * p, h_r * p]
    for p in p_values:
        r_min = l_r * p
        r_max = h_r * p
        r_values = np.random.uniform(r_min, r_max, number_of_r)
        
        # For each r, a 2x2 transition matrix is created and stored
        for r in r_values:
            matrix = np.array([[1 - p, p], [r, 1 - r]])
            matrices.append(matrix)

    # Step 2: Creation of a DataFrame of MTTR/MTTF values and extract parameters
    MTTR_MTTF = pd.DataFrame({
        'p': [1 - m[0, 0] for m in matrices],  # Original p extraction
        'r': [m[1, 0] for m in matrices]       # r extraction
    })
    
    # p and r values are stored as tuples for later use
    matrix_params = list(zip(MTTR_MTTF['p'], MTTR_MTTF['r']))

    # Step 3: configuration rows ae generated
    rows_per_k = int(number_of_rows_for_each_line / number_of_class)  # Rows per machine-count (k)
    k_values = range(2, range_of_machines + 1)  # Range of different numbers of machines (k) from k=2 to k= K
    data = []

    for k in k_values:
        unique_combinations = set()  # To avoid duplicate configurations
        
        # Generation of configurations until the desired count is reached
        while len(unique_combinations) < rows_per_k:
            # Buffer capacities (k-1 values)are generated , using pre-defined values
          #  custom_values = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
           # buffer_capacity = np.random.choice(custom_values, size=k - 1).tolist()
            buffer_capacity = np.random.randint(1, 26, size=k - 1).tolist()
            buffer_tuple = tuple(buffer_capacity)

            # Randomly 'k' transition matrices are selected with their parameters
            selected_indices = np.random.choice(len(matrices), k)
            selected_params = tuple(matrix_params[idx] for idx in selected_indices)

            # A key is created to ensure uniqueness of configuration
            key = (buffer_tuple, selected_params)

            # If configuration is unique, it is added to the set and recorded 
            if key not in unique_combinations:
                unique_combinations.add(key)

                # Each selected matrix is formated as a string with 4 decimal places
                probabilities = [
                    f"[{matrices[idx][0, 0]:.4f}, {matrices[idx][0, 1]:.4f}; {matrices[idx][1, 0]:.4f}, {matrices[idx][1, 1]:.4f}]"
                    for idx in selected_indices
                ]

                # A one row is created and it consists of: [# machines, buffer capacities, list of matrices, and placeholders for unused matrix columns]
                row = [k, str(buffer_capacity).replace(" ", "")] + probabilities + ['None'] * (range_of_machines - len(probabilities))
                data.append(row)

    # Step 4: All configuration rows are converted into a DataFrame
    column_names = ['Number of Machines', 'Buffers Capacities'] + [f'Matrix_{i}' for i in range(1, range_of_machines + 1)]
    input_configurations = pd.DataFrame(data, columns=column_names)

    return input_configurations


# Application of function to generate configurations with specified parameters
# This example targets average efficiency (0.90 <= r / (r + p) < 0.97)
input_configurations_1_25_buffer_capacity_50_97_efficiency = generate_input_configurations(
    number_of_p=15,
    number_of_r=10,
    p_maximum_range=0.0309,
    l_r=1,
    h_r=32.33,
    number_of_rows_for_each_line=45000,
    number_of_class=1)

# ----------------- Saving the output to CSV files --------------------

import os  # OS module for directory operations

# A folder name is defined and created if it doesn't exist
output_dir = 'input_configurations_1_25_buffer_capacity_50_97_efficiency2'
os.makedirs(output_dir, exist_ok=True)

# Configurations are grouped by number of machines and save the first 45,000 rows of each group to separate CSV files
for number, df in input_configurations_1_25_buffer_capacity_50_97_efficiency .groupby('Number of Machines'):
    # File path for current group is defined
    file_path = os.path.join(output_dir, f'df_Buffer_set_{number}_machines.csv')

    # First 45,000 rows are saved, and missing values are filled with 'None', and index is reset
    df.head(45000).fillna(value='None').reset_index(drop=True).to_csv(file_path, index=False)

# Confirmation message
print(f"Saved DataFrames for each group based on 'Number of Machines' '{output_dir}' folder.")


# 2. Simulation

## Generated data Files are then simulated through the SPL_Matlab_Simulation.m File

# 3. Exploratory data Analysis (EDA) on the SPL Data 

### Importing the needed libraries for Exploratory data Analysis

In [ ]:
import re  # For Regular expressions 
import gc  # Triggeirimh garbage collection and free memory
import os  # Interacting with the operating system 
import csv  # Reading/Writing CSV files
import shutil # For zipping the folder
import numpy as np  # Numerical operations on arrays
import pandas as pd  # Handling data structures like DataFrames (tabular data)
import seaborn as sns  # Data visualization library based on matplotlib
import warnings  # Warning messages
warnings.filterwarnings("ignore")  # Suppresses all warning messages
from itertools import zip_longest  
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde  # Kernel Density Estimation for distribution estimation


# Loading the data
load the csv files from the folder and concatenates them into one csv file.
The file conists of the following:

Input data: conisiting of

1) Number of machines
2) Buffer capacities list
3) Machine transtion probabilities matrices

Output data: conisiting of:

1. Average Throughput
2. Standered deviation of Throughput
3. Confindence Interval of  Throughput
4. Confindence Interval of Stander ed deviation
5. Average Buffers Levels
7. Mean Time to starvation for each replicate of simulation (mTTs)

In [ ]:
import os
import pandas as pd

def load_and_concat(folder_path):
    """
    Reads all CSV files in a folder and concatenates them
    while keeping missing values as None/NaN.
    """
    files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    
    if not files:
        raise ValueError("No CSV files found in the folder.")

    dataframes = []

    for file in files:
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)
        dataframes.append(df)

    combined_df = pd.concat(dataframes, ignore_index=True)

    # No filling -> keep NaN/None
    return combined_df
#Printing the name of columns and data types, and descriptve statics    
def describe_data(data):
    print(f"\nDataset Information: \n")
    data.info()
    print(f"\nDescriptive Statistics :\n")
    return data.describe().transpose().reset_index()

In [ ]:
folder = r"D:\buffers\SPL Configurations Files\Unpacked SPL Matrices and Buffers data files"
data = load_and_concat(folder)

In [ ]:
pd.set_option("display.max_rows", None)  # Show all rows
pd.set_option("display.max_columns", None)  # Show all columns
pd.set_option("display.width", None)  # Auto-detect the display width
pd.set_option("display.max_colwidth", None)  # Show full content of each cell
data.sample(5)

In [ ]:
# Reset display options to default
pd.reset_option("display.max_rows")
pd.reset_option("display.max_columns")
pd.reset_option("display.width")
pd.reset_option("display.max_colwidth")
describe_data(data)


### Defined functions needed for Unpakcing the data

In [ ]:

def Unpacking_Buffers_Capacities_list(df):
    """Unpacking the data from the lists for the buffers capacity list so each value has its own column in the dataframe 
Unpacking procesdure consits of the following steps:
First step: Determine the buffer list length
Second step: Create columns to mach the buffer max size and name them based on thier possition
Third step: Pad the shorter list ith None for the time being
[2,4]-->2,4,None
[2,4,6]-->2,4,6,None
[2,4,6,18]-->2,4,6,18
Fourth step: Convert the padded list into dataprame and concate them into the original dataframe 
Fifith step: Drop the original 'Buffers Capacities' list column
"""
    df = df.copy()    # Ensuring we are working with a copy of the DataFrame 
    # Phase One: Unpakcing Buffer Capacities List
    # Converting string representation of lists into actual lists
    df["Buffers Capacities"] = df["Buffers Capacities"].apply(lambda x: eval(x) if isinstance(x, str) else x )
    # Getting the maximum length of the lists in the "Buffers Capacities" column
    max_len = df["Buffers Capacities"].apply(len).max()
    # Creating column names to match the maximum length
    buffer_cols = ["Buffer_" + str(i+1) for i in range(max_len)]
    # Pading shorter lists with None 
    padded_lists = df["Buffers Capacities"].apply(lambda x: x + [None] * (max_len - len(x)))
    # Converting the padded lists into a DataFrame
    buffer_matrix = pd.DataFrame(padded_lists.tolist(), index=df.index, columns=buffer_cols)
    # Concatenating the exploded buffer columns with the original DataFrame
    df = pd.concat([df, buffer_matrix], axis=1)
    # Droping the original "Buffers Capacities" column
    df = df.drop(columns=["Buffers Capacities"])
    return df
    
  

def Unpacking_transition_probability_matrices(df):
    """Unpacking the data from the lists transition probabilities matrices with eac matrix having a
    string representations of 2×2 matrices,so each value has its own column in the dataframe 
Unpacking procesdure consits of the following steps:
First step: Look for columns that start with 'Matrix_'
Second step: Strip the Square bracketsand split the matrices based on the comma','
and split the values based on the comma ','
1. Make sure x is a string and not "None", and split it using "; " to separate the rows.
2. Make sure each row is stripped of brackets ("[]") and split by ", " to extract float values.
3. If x is "None" or not a string, a default matrix [[0, 0], [0, 0]] is assigned.
Third step: Ensure each row has exactly 4 values (2x2 matrix)
Fourth step: Split the 2×2 Matrix into separate columns with these new names (Puu,Pud,Pdu,Pdd)  added to the 'Matrix_i_'
Fifith step: Drop the original 'Matrix_i' columns

[0.95,0.05;0.2,0.8]-->0.95,0.05,0.2,0.8
"""
    df = df.copy()    # Ensuring we are working with a copy of the DataFrame 
      # Phase Two: Explode Probabilities_X columns
    for col in df.columns:
        if col.startswith("Matrix_"):
            # Split the string into nested lists
            df[col] = df[col].apply(
                lambda x: [
                    [float(val) for val in part.strip("[]").split(", ")]
                    for part in x.split("; ")
                ] if x != "None" and isinstance(x, str) else [[None, None], [None, None]])
            # Ensure each row has exactly 4 values (2x2 matrix)
            df[col] = df[col].apply(lambda x: x if len(x) == 2 and all(len(part) == 2 for part in x) else None)# [[None, None], [None, None]])

            # Explode into separate columns with new naming convention
            prob_cols = [f"{col}_Puu", f"{col}_Pud", f"{col}_Pdu", f"{col}_Pdd"]
            df[prob_cols] = df[col].apply(lambda x: pd.Series([x[0][0], x[0][1], x[1][0], x[1][1]]))
            # Drop the original column
            df = df.drop(columns=[col])

    # Step 3: Replace "None" with NaN
    df.replace("None", np.nan, inplace=True)
    return df    

In [ ]:
unpacked_Buffers_data = Unpacking_Buffers_Capacities_list(data)
unpacked_data= Unpacking_transition_probability_matrices(unpacked_Buffers_data)

pd.set_option("display.max_rows", None)  # Show all rows
pd.set_option("display.max_columns", None)  # Show all columns
pd.set_option("display.width", None)  # Auto-detect the display width
pd.set_option("display.max_colwidth", None)  # Show full content of each cell
unpacked_data.sample(5)

In [ ]:
pd.reset_option("display.max_rows")
pd.reset_option("display.max_columns")
pd.reset_option("display.width")
pd.reset_option("display.max_colwidth")
del data
del unpacked_Buffers_data

## Data Cleaning
This process conists of chekcking for missing data, duplicates, dealing with this missing data by filling zeros for the buffers and removing outliers.

In [ ]:

def check_for_duplication(data, exclude_columns):
    # Drop specified columns before checking for duplicates
    filtered_data = data.drop(columns=exclude_columns, errors='ignore')

    # Count duplicate rows (excluding specified columns)
    num_duplicates = filtered_data.duplicated(keep=False).sum()

    print(f"Number of duplicate rows (excluding specified columns): {num_duplicates}")
def check_for_missing_values(data):
    """Return a dictionary with column names and the count of missing values."""
    missing_counts = data.isnull().sum()  # Count missing values per column
    missing_counts = missing_counts[missing_counts > 0]  # Filter only columns with missing values
    missing_dict = missing_counts.to_dict()  # Convert to dictionary
    print("\nColumns with Missing Values and Their Counts:")
    for col, count in missing_dict.items():
        print(f"{col}: {count} missing values")

def replace_nan_in_range(df, col_names, replace_value):
    df = df.copy()     # Ensuring the DataFrame is not modified in place
    df[col_names] = df[col_names].fillna(replace_value)     # Getting the column names in the specified slice
    return df    

### Checking for duplicates and missing values

In [ ]:
exclude_columns = [
    'Avg_NP1', 'Std_NP1',
    'CI_NP_1', 'CI_NP_2',
    'CI_Std_NP_1', 'CI_Std_NP_2'
]

# Dynamically add mTTS_1 to mTTS_99
exclude_columns += [f"mTTS_{i}" for i in range(1, 6)]

# Dynamically add Avg_BL_1 to Avg_BL_99
exclude_columns += [f"Avg_BL_{i}" for i in range(1, 20)]

check_for_duplication(data, exclude_columns)
check_for_missing_values(data)

### Removing Unnecessary Columns 
Here only the Throughput (Avg_NP1) is kept from the output columns and the others columns are dropped

In [ ]:
columns_to_drop = [col for col in data.columns 
                   if col.startswith("Std_NP") 
                   or col.startswith("CI_NP") 
                   or col.startswith("CI_Std_NP") 
                   or col.startswith("mTTS_") 
                   or col.startswith("Avg_BL_")]

Removed_columns_data = data.drop(columns=columns_to_drop, axis=1)


### Replacing Nan values by zeros to unifie the data (Structured data)

In [ ]:

def get_columns_by_pattern(df, pattern):
    """Return all column names in df that match a regex pattern."""
    return [col for col in df.columns if re.match(pattern, col)]

# Replace NaNs in all 'Buffer_n' columns
buffer_cols = get_columns_by_pattern(Removed_columns_data, r'^Buffer_\d+$')
df_replaced_Buffers = replace_nan_in_range(Removed_columns_data, buffer_cols, replace_value=0)

# Replace NaNs in all 'Matrix_n_xyz' columns
matrix_cols = get_columns_by_pattern(df_replaced_Buffers, r'^Matrix_\d+_P[ud]{2}$')
df_masked_machines_with_zeros_ = replace_nan_in_range(df_replaced_Buffers, matrix_cols, replace_value=0)


df_masked_machines_with_zeros_

### Description of the flattened data

In [ ]:
describe_data(df_masked_machines_with_zeros_)

# Filtering dataframes based on Number of machines
Creating datasets for different configurations of Serial Production Lines based on the number of machines

k = 2--> group_2 [ 'Number of Machines',

                   'Matrix_1_Puu','Matrix_1_Pud','Matrix_1_Pdu','Matrix_1_Pdd', ,'Buffer_1',
      
                   'Matrix_2_Puu','Matrix_2_Pud','Matrix_2_Pdu','Matrix_2_Pdd',
      
                   'Avg_NP1']
k = 3--> group_3 [ 'Number of Machines',

                   'Matrix_1_Puu','Matrix_1_Pud','Matrix_1_Pdu','Matrix_1_Pdd','Buffer_1',
      
                   'Matrix_2_Puu','Matrix_2_Pud','Matrix_2_Pdu','Matrix_2_Pdd','Buffer_2',
      
                   'Matrix_3_Puu','Matrix_3_Pud','Matrix_3_Pdu','Matrix_3_Pdd',
      
                   'Avg_NP1']
and so on...

In [ ]:
# --- Get sorted unique machine counts ---
unique_machine_counts = sorted(df_masked_machines_with_zeros_['Number of Machines'].dropna().unique())

# --- Group by machine count ---
group_dict = {
    num: df_masked_machines_with_zeros_[df_masked_machines_with_zeros_['Number of Machines'] == num].copy()
    for num in unique_machine_counts}

# --- Function to determine which columns to drop ---
def get_columns_to_drop(num_machines, max_machines):
    columns_to_drop = []

    # Drop buffer columns beyond current machine count
    columns_to_drop.extend([f'Buffer_{i}' for i in range(int(num_machines), int(max_machines))])

    # Drop matrices beyond current machine count
    for i in range(int(num_machines + 1), int(max_machines) + 1):
        columns_to_drop.extend([
            f'Matrix_{i}_Puu', f'Matrix_{i}_Pud', f'Matrix_{i}_Pdu', f'Matrix_{i}_Pdd',
        ])
    
    return columns_to_drop
def filter_and_reorder_columns(df, num_machines, max_machines, target_col="Avg_NP1"):
    """
    Keeps relevant matrices/buffers for num_machines and reorders them.
    Retains 'Number of Machines' and the target column.
    Buffers go from 1 to (num_machines - 1).
    """
    cols_to_keep = ["Number of Machines", target_col]  # Always keep these

    for i in range(1, int(num_machines) + 1):
        # Add matrix columns if they exist
        for suffix in ["Puu", "Pud", "Pdu", "Pdd"]:
            col_name = f"Matrix_{i}_{suffix}"
            if col_name in df.columns:
                cols_to_keep.append(col_name)

        # Add buffer column only if i < num_machines
        if i < num_machines:
            buffer_col = f"Buffer_{i}"
            if buffer_col in df.columns:
                cols_to_keep.append(buffer_col)

    df_filtered = df[cols_to_keep].copy()
    return df_filtered

# Apply to all machine counts
df_filtered_dict = {}
max_machines = max(unique_machine_counts)

for num_machines in unique_machine_counts:
    df_filtered_dict[num_machines] = filter_and_reorder_columns(
        group_dict[num_machines], num_machines, max_machines, target_col="Avg_NP1")


# Saving SPL data based on Length

In [ ]:
import shutil # File operations like making archives
from IPython.display import FileLink  # Generating a downloadable link in Jupyter environments

save_folder = 'df_filtered_dictionary_90_97_eff_5_to_50_buffer' # Folder where the CSVs will be saved
os.makedirs(save_folder, exist_ok=True) # Create the folder if it doesn't already exist

# Save each DataFrame in the dictionary to a separate CSV file
for key, df in df_filtered_dict.items():
    file_path = os.path.join(save_folder, f"{key}.csv") # Creating file path for each CSV
    df.to_csv(file_path, index=False) # Saving DataFrame without row indices
folder_to_zip = 'df_filtered_dictionary_90_97_eff_5_to_50_buffer'  # Folder to be zipped
zip_file_name = 'df_filtered_dictionary_90_97_eff_5_to_50_buffer.zip'  # Output zip file name

# Create a zip archive of the folder
shutil.make_archive(base_name=folder_to_zip, format='zip', root_dir=folder_to_zip)
FileLink('df_filtered_dictionary_90_97_eff_5_to_50_buffer.zip') # Generating a clickable download link for the zip file 

# 4. SPL Data Feature Engineering and Selection

In [ ]:
import os  # Interacting with the operating system 
import csv  # Reading/Writing CSV files
import pandas as pd  # Handling data structures like DataFrames (tabular data)
import zipfile  # For working with ZIP archives (not used here, but might be needed later)
import re  # For Regular expressions 
import math  # Access to mathematical functions 
import numpy as np  # Numerical operations on arrays
import matplotlib.pyplot as plt  # not used here but common with pandas
from sklearn.decomposition import PCA   # Principal Component Analysis — for dimensionality reduction and feature compression.
from sklearn.utils.extmath import randomized_svd #  Efficient randomized algorithm to compute truncated Singular Value Decomposition (SVD).
from sklearn.feature_selection import mutual_info_regression # Computes mutual information scores between features and target for feature selection.
import shutil  # For zipping the folder
from IPython.display import FileLink # create clickable links



# Loading the data

In [ ]:
import zipfile
import os
import pandas as pd

load_folder =r'D:\buffers\SPL Configurations Files\Rearranged SPL Feature Sselected dataframe dictionary'
# Read and sort CSVs by the numeric value of the filename number
df_filtered_dict = {
    k: v for k, v in sorted( # Sort dictionary by keys (filenames as integers)
        {
            int(os.path.splitext(filename)[0]): # Extracting integer from filename (remove ".csv")
            pd.read_csv(os.path.join(load_folder, filename)) # Reading the CSV into a DataFrame
            for filename in os.listdir(load_folder) # Looping through all files in the folder
            if filename.endswith('.csv') # Only .csv files are considered
        }.items()  # Converting the dictionary to items for sorting
    )
}

# Feature Enginerring 
This process consists of selecting, manipulating and transforming raw data into features that can be used in supervised learning. This section inludes steps of producing features from the domain using expertise of the domain:

1. Efficiency of the machines e= r/r+p
2. Variability of the machines v=e(1- e)*(1+ρ)/(1-ρ)
3. Total_buffers_capacities TB=b1+b2+b3+....
4. Efficiency of the line by multiplying each machine efficiency e_1 * e_2 * ...

In [ ]:
def detect_num_machines(df):
    import re
    machine_nums = []
    for col in df.columns:
        match = re.match(r"Matrix_(\d+)_", col)
        if match:
            machine_nums.append(int(match.group(1)))
    return max(machine_nums) if machine_nums else 0
    
def calculate_variability(df):
    num_machines = detect_num_machines(df) # Automatically detect how many machines (based on matrix columns)
    new_cols = {}

    for i in range(1, num_machines + 1): # Looping over each machine
        efficiency_col = f'Matrix_{i}_Efficiency'
        variability_col = f'Matrix_{i}_Variability'
        prob_pdu_col = f'Matrix_{i}_Pdu'
        prob_pud_col = f'Matrix_{i}_Pud'
        matrix_cols = [f'Matrix_{i}_Puu', prob_pud_col, prob_pdu_col, f'Matrix_{i}_Pdd']

        if all(col in df.columns for col in matrix_cols + [efficiency_col]): # Check all necessary columns exist
            # Convert necessary columns to float
            df_subset = df[matrix_cols + [efficiency_col]].astype(float) # Ensure numeric type
            matrix_sum = df_subset[matrix_cols].sum(axis=1)
            denominator = df_subset[prob_pdu_col] + df_subset[prob_pud_col]
            p = 1 - df_subset[prob_pdu_col] - df_subset[prob_pud_col] # Internal probability variable
            # Create a mask for valid rows
            valid_mask = (matrix_sum != 0) & (denominator != 0) # Avoid division by zero
            # Initialize variability column with zeros
            variability = np.zeros(len(df), dtype=float) # Initialize result
            # Apply the variability formula where valid
            efficiency_vals = df_subset[efficiency_col]
            p_vals = p
            variability[valid_mask] = (
                efficiency_vals[valid_mask] * (1 - efficiency_vals[valid_mask]) *
                ((1 + p_vals[valid_mask]) / (1 - p_vals[valid_mask]))
            )
            new_cols[variability_col] = variability # Store result
        else:
            print(f"⚠️ Missing columns for Machine {i}: {matrix_cols + [efficiency_col]}")
    # Create DataFrame from new columns and concatenate with original
    df_new_cols = pd.DataFrame(new_cols, index=df.index)  # New columns as DataFrame
    final_df = pd.concat([df, df_new_cols], axis=1)  # Merge with original
    return final_df

def calculate_efficiency(df):
    num_machines = detect_num_machines(df)
    for i in range(1, num_machines + 1):
        prob_pdu_col = f'Matrix_{i}_Pdu'
        prob_pud_col = f'Matrix_{i}_Pud'
        efficiency_col = f'Matrix_{i}_Efficiency'

        if prob_pdu_col in df.columns and prob_pud_col in df.columns:
            # Ensure columns are float for accurate division
            df[prob_pdu_col] = df[prob_pdu_col].astype(float)
            df[prob_pud_col] = df[prob_pud_col].astype(float)
            denominator = df[prob_pdu_col] + df[prob_pud_col]
            zero_mask = (denominator == 0)  # Handle divide-by-zero
            # Default to zero efficiency when denominator is zero
            efficiency = df[prob_pdu_col] / denominator
            efficiency[zero_mask] = 0.0
            # Assign to new column
            df[efficiency_col] = efficiency
        else:
            print(f"⚠️ Missing columns for Machine {i}: {prob_pdu_col}, {prob_pud_col}")
    return df
    
def multiply_nonzero_efficiencies(df):
    num_machines = detect_num_machines(df)
    product_col = 'Efficiency_Product'
    efficiency_cols = [f'Matrix_{i}_Efficiency' for i in range(1, num_machines + 1) if f'Matrix_{i}_Efficiency' in df.columns]
    df[product_col] = 1.0
    for col in efficiency_cols:
        temp = df[col].replace(0, 1) # Avoid zeroing the product
        mask = df[col] != 0
        df.loc[mask, product_col] *= temp   # Only multiply non-zero values
    return df

def Total_buffers_capacities(df):
    """
    Sums all columns that match the pattern 'Buffer_X' and creates a new column 'Total_Buffer'.
    Returns the 'Total_Buffer' column.
    """
    buffer_cols = [col for col in df.columns if col.startswith("Buffer_")]
    df["Total_Buffer"] = df[buffer_cols].sum(axis=1)  # Sum all Buffer_X
    return df["Total_Buffer"]  # Return only the new column

In [ ]:
df_Feature_Engineered={}
for num_machines, df in df_filtered_dict.items():
    print("Processing SPL configureation with number of machines:",num_machines) 
    Total_buffers_capacities(df)
    df_process=calculate_efficiency(df)   
    df_process=calculate_variability(df_process)  
    df_process=multiply_nonzero_efficiencies(df_process)
    df_Feature_Engineered[num_machines] = df_process

# Rearragngment of the data Coulmns
1. First column is Number of Machines
2. Then the Buffers
3. Then for each machine its Efficiency ,Variability,.

In [ ]:

rearranged_data_frame_dic={}
for k, group in df_filtered_dictionary.items():
    max_machines = group['Number of Machines'].max()
    num_buffers = max_machines - 1
    
    # Step 2: Create the list of column names for buffers, matrices, and other columns
    buffer_columns = [f'Buffer_{i}' for i in range(1, num_buffers + 1)]
    matrix_columns = [f'Matrix_{i}_Efficiency' for i in range(1, max_machines + 1)] + \
                     [f'Matrix_{i}_Variability' for i in range(1, max_machines + 1)]
    # Step 3: Create the desired column order list
    desired_column_order = ['Number of Machines'] + buffer_columns + matrix_columns + \
                           ['Efficiency_Product','Total_Buffer','Avg_NP1']
    
    # Step 4: Rearrange the DataFrame columns according to the desired order
    rearranged_data_frame_dic[k] = group[desired_column_order]

# Save the Data

In [ ]:

save_folder = 'rearranged_feature_selected_data_frame_dic_50_97_eff_1_to_25_buffer'
os.makedirs(save_folder, exist_ok=True)

# Save each DataFrame
for key, df in rearranged_data_frame_dic.items():
    file_path = os.path.join(save_folder, f"{key}.csv")
    df.to_csv(file_path, index=False)
folder_to_zip = 'rearranged_feature_selected_data_frame_dic_50_97_eff_1_to_25_buffer'
zip_file_name = 'rearranged_feature_selected_data_frame_dic_50_97_eff_1_to_25_buffer.zip'

# Create a zip archive
shutil.make_archive(base_name=folder_to_zip, format='zip', root_dir=folder_to_zip)
FileLink('rearranged_feature_selected_data_frame_dic_50_97_eff_1_to_25_buffer.zip')    

# 5. Traditional Machine Learning Models on the featured SPL Data 

In [ ]:
import os  # Interacting with the operating system 
import pandas as pd  # Handling data structures like DataFrames (tabular data)
import pickle  # For serializing and de-serializing Python objects
import numpy as np  # Numerical operations on arrays
import matplotlib.pyplot as plt # For creating plots and visualizations (e.g., heatmaps, subplots)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, ShuffleSplit  
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler  
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score  # Regression metrics
import shutil # For zipping the folder
from sklearn import metrics  # General metrics module 
#Regression models from sklearn
#from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet  
#from sklearn.tree import DecisionTreeRegressor  # Decision tree-based regression
#from sklearn.ensemble import RandomForestRegressor  # Ensemble method using multiple decision trees
#import xgboost as xgb  # Extreme Gradient Boosting (efficient tree-based ML library)
import lightgbm as lgb  # LightGBM - Gradient boosting by Microsoft (faster and more memory efficient)
#from sklearn.neighbors import KNeighborsRegressor  # KNN regression based on feature similarity
import shap 
import time

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "figure.dpi": 300,
    "savefig.dpi": 300,
})

# Better embedded fonts for publication PDFs
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

## Traditional ML models
### ML Modles for each configuration line (alone)


In [ ]:

def zip_folder(folder_path, zip_output_path):
    if os.path.exists(folder_path):
        shutil.make_archive(zip_output_path.replace(".zip", ""), 'zip', folder_path)
        print(f"✅ Zipped folder: {folder_path} → {zip_output_path}")
    else:
        print(f"⚠️ Folder not found: {folder_path}")

def plot_sampled_predictions(y_true, y_pred, dataset_name="", save_prefix=None, Line_sample=50,Scatter_sample=10, save_dir="plots"):
    n=Line_sample
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    x_sampled = np.arange(len(y_true))[::n]
    y_true_sampled = y_true[::n]
    y_pred_sampled = y_pred[::n]

    # Ensure save directory exists
    if save_prefix and save_dir:
        os.makedirs(save_dir, exist_ok=True)

    # --- Line Plot ---
    plt.figure(figsize=(8, 6))
    plt.figure(figsize=(8, 6))
    plt.plot(x_sampled, y_true_sampled, label="True")#, marker='o', linestyle='-')
    plt.plot(x_sampled, y_pred_sampled, label="Predicted", alpha=0.75)#, marker='x', linestyle='--'
    plt.xlabel("Sample (every {}th point)".format(n))
    plt.ylabel("Average Throughput")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    if save_prefix:
        plt.savefig(os.path.join(save_dir, f"{save_prefix}_{dataset_name}_line_sampled.png"))
    plt.show()

    # --- Scatter Plot ---
    n=Scatter_sample
    x_sampled = np.arange(len(y_true))[::n]
    y_true_sampled = y_true[::n]
    y_pred_sampled = y_pred[::n]
    plt.figure(figsize=(8, 6))
    plt.scatter(x_sampled, y_true_sampled, label="True", s=10)
    plt.scatter(x_sampled, y_pred_sampled, label="Predicted", s=10, alpha=0.7)
    plt.xlabel("Sample (every {}th point)".format(n))
    plt.ylabel("Average Throughput")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    if save_prefix:
        plt.savefig(os.path.join(save_dir, f"{save_prefix}_{dataset_name}_scatter_sampled.png"))
    plt.show()

def generate_shap_plots(best_model, X_train, X_test,
                        feature_names, model_name,
                        dataset_name, save_dir="plots"):

    shap_dir = os.path.join(save_dir, model_name, "shap")
    os.makedirs(shap_dir, exist_ok=True)

    scaler = best_model.named_steps['scaler']
    model = best_model.named_steps['model']

    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    X_sample = X_test_scaled[:300]

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)

    # =========================================
    # FIX FEATURE NAMES (scalable)
    # =========================================
    corrected_feature_names = [f.replace("Matrix_", "Machine_")for f in feature_names]

    # =========================
    # Mean absolute SHAP
    # =========================
    mean_abs_shap = np.abs(shap_values).mean(axis=0)

    shap_df = pd.DataFrame({"feature": corrected_feature_names,
                            "importance": mean_abs_shap,
                            "dataset": dataset_name,
                            "model": model_name})

    # ===== Summary Plot =====
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values,X_sample,feature_names=corrected_feature_names,show=False)

    plt.tight_layout()
    plt.savefig(os.path.join(shap_dir, f"{dataset_name}_summary.png"),dpi=300,bbox_inches='tight')
    plt.close()

    # ===== Bar Plot =====
    plt.figure(figsize=(8, 6))
    shap.summary_plot(shap_values,X_sample,feature_names=corrected_feature_names,plot_type="bar",show=False)

    plt.tight_layout()
    plt.savefig(os.path.join(shap_dir, f"{dataset_name}_bar.png"),dpi=300,bbox_inches='tight')
    plt.close()

    print(f"✅ SHAP plots saved for {dataset_name}")

    return shap_df


def run_model_with_gridsearch(model,model_name,param_grid,data_dict,target_col='Avg_NP1',output_prefix='model_output'):
    scoring = {'MSE': 'neg_mean_squared_error',
               'RMSE': 'neg_root_mean_squared_error',
               'MAE': 'neg_mean_absolute_error',
               'R2': 'r2' }
    all_shap_results = []
    results = []
    all_cv_results = []
    combined_results = {}
    best_metrics_list = []
    timing_results = {}
    for dataset_name, df in data_dict.items():
        print(f"\n### Processing dataset: {dataset_name}")

        X = df.drop(columns=[target_col])
        y = df[target_col]

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        pipe = Pipeline([('scaler', StandardScaler()),
                         ('model', model)])

        grid = GridSearchCV(estimator=pipe,param_grid=param_grid,scoring=scoring,
                            refit='R2',cv=5,n_jobs=-1,return_train_score=True)

        grid.fit(X_train, y_train)

        cv_results_df = pd.DataFrame(grid.cv_results_)
        cv_results_df['dataset'] = dataset_name
        cv_results_df['model'] = model_name

        # Evaluate all grid candidates on test set
        test_metrics = {'test_mse': [],
                        'test_mae': [],
                        'test_r2': [],
                        'relative_mse': [],
                        'relative_mae': [],
                        'relative_absolute_error': [],
                        'relative_squared_error': [],
                        'Test_MAPE': []}
        best_params = grid.best_params_

        for i in range(len(cv_results_df)):
            params = cv_results_df.loc[i, 'params']
            if params != best_params:
                continue
            pipe.set_params(**params)
            start = time.time()
            pipe.fit(X_train, y_train)
            fit_time = time.time() - start
            y_pred = pipe.predict(X_test)

            mse = mean_squared_error(y_test, y_pred)
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            relative_mse = mse / np.mean(y_test ** 2)
            relative_mae = mae / np.mean(np.abs(y_test))
            rae = np.sum(np.abs(y_test - y_pred)) / np.sum(np.abs(y_test - np.mean(y_test)))
            rse = np.sum((y_test - y_pred) ** 2) / np.sum((y_test - np.mean(y_test)) ** 2)
            MAPE = np.mean(np.abs(y_pred - y_test) / (y_test))  # prevent div by zero

            test_metrics['test_mse'].append(mse)
            test_metrics['test_mae'].append(mae)
            test_metrics['test_r2'].append(r2)
            test_metrics['relative_mse'].append(relative_mse)
            test_metrics['relative_mae'].append(relative_mae)
            test_metrics['relative_absolute_error'].append(rae)
            test_metrics['relative_squared_error'].append(rse)
            test_metrics['Test_MAPE'].append(MAPE)
            test_metrics.setdefault('fit_time', []).append(fit_time)
            best_metrics_df = pd.DataFrame({
                'dataset': [dataset_name],
                'model': [model_name],
                'best_params': [grid.best_params_],
                'test_mse': [mse],
                'test_mae': [mae],
                'test_r2': [r2],
                'relative_mse': [relative_mse],
                'relative_mae': [relative_mae],
                'relative_absolute_error': [rae],
                'relative_squared_error': [rse],
                'fit_time': [fit_time],
                'Test_MAPE': [MAPE]})
       # for metric_name, values in test_metrics.items():
        #    cv_results_df[metric_name] = values
        best_metrics_list.append(best_metrics_df)
        all_cv_results.append(cv_results_df)
       # cv_results_df[metric_name] = values
        # ---- Best model evaluation ----
        best_model = grid.best_estimator_
        # ===== SHAP =====
        shap_df = generate_shap_plots(best_model=best_model,X_train=X_train,X_test=X_test,
                                      feature_names=X.columns,model_name=model_name,dataset_name=dataset_name)

        all_shap_results.append(shap_df)

        #y_pred = best_model.predict(X_test)
        # ---- inference timing ----
        n_samples = len(X_test)
        # warm-up (important)
        _ = best_model.predict(X_test.iloc[:10])
        
        start = time.perf_counter()
        y_pred = best_model.predict(X_test)
        end = time.perf_counter()
        # compute FIRST
        inference_time_total = end - start
        inference_time_per_sample = inference_time_total / n_samples

        timing_results[dataset_name] = {"n_samples": n_samples,
                                        "inference_time_total_sec": inference_time_total,
                                        "inference_time_per_sample_sec": inference_time_per_sample,
                                        "fit_time_sec": fit_time,}

        inference_time_total = end - start
        inference_time_per_sample = inference_time_total / n_samples
        y_test_arr = np.asarray(y_test).ravel()
        y_pred_arr = np.asarray(y_pred).ravel()
        combined_results[dataset_name] = (y_test_arr, y_pred_arr)
        
        # Plot and save
        plot_sampled_predictions( y_test_arr, y_pred_arr,dataset_name=dataset_name,save_prefix=f"{model_name}_{dataset_name}",
            save_dir=os.path.join("plots", model_name),Line_sample=50,Scatter_sample=10)

        # Save best model
        model_folder = os.path.join("saved_models", model_name)
        os.makedirs(model_folder, exist_ok=True)
        model_filename = f"{model_name}_{dataset_name}_best.pkl"
        model_path = os.path.join(model_folder, model_filename)
        with open(model_path, "wb") as f:
            pickle.dump(best_model, f)
        print(f"✅ Saved best model to: {model_path}")

        # Save best metrics
        mse = mean_squared_error(y_test_arr, y_pred_arr)
        mae = mean_absolute_error(y_test_arr, y_pred_arr)
        r2 = r2_score(y_test_arr, y_pred_arr)
        relative_mse = mse / np.mean(y_test_arr ** 2)
        relative_mae = mae / np.mean(np.abs(y_test_arr))
        rae = np.sum(np.abs(y_test_arr - y_pred_arr)) / np.sum(np.abs(y_test_arr - np.mean(y_test_arr)))
        rse = np.sum((y_test_arr - y_pred_arr) ** 2) / np.sum((y_test_arr - np.mean(y_test_arr)) ** 2)
        MAPE = np.mean(np.abs(y_pred_arr - y_test_arr) / (y_test_arr))  # prevent div by zero

        results.append({
            'dataset': dataset_name,
            'model': model_name,
            'best_params': grid.best_params_,
            'test_mse': mse,
            'test_mae': mae,
            'test_r2': r2,
            'relative_mse': relative_mse,
            'relative_mae': relative_mae,
            'relative_absolute_error': rae,
            'relative_squared_error': rse,
            'Test_MAPE':MAPE,
            'fit_time': fit_time,
            'inference_time_total': inference_time_total,
            'inference_time_per_sample': inference_time_per_sample,
            'n_test_samples': n_samples})
    # ==========================================
    # Aggregate SHAP across all sequence lengths
    # ==========================================    
    all_shap_df = pd.concat(all_shap_results, ignore_index=True)
    all_shap_df["feature"] = (all_shap_df["feature"].str.replace("Matrix_", "Machine_", regex=False))
    # Save raw shap values
    all_shap_df.to_csv(f"{output_prefix}_all_shap_values.csv",index=False)   
    # ==========================================
    # Fix feature names before aggregation
    # ==========================================
    # Mean importance across datasets
    global_shap = (all_shap_df.groupby("feature")["importance"].mean().sort_values(ascending=False).reset_index())    
    # Save aggregated importance
    global_shap.to_csv(f"{output_prefix}_global_shap.csv",index=False)    
    # ==========================================
    top_features = (global_shap["feature"].head(5).tolist())
    TOP_N = 10
    global_shap_top = global_shap.head(TOP_N)

    plt.figure(figsize=(8, 6))
    
    plt.barh(global_shap_top["feature"],global_shap_top["importance"])
    
    plt.gca().invert_yaxis()
    
    plt.xlabel("Mean |SHAP value|")
    plt.ylabel("Feature")
    
    plt.tight_layout()
    
    save_base = os.path.join("plots", model_name,f"{model_name}_GLOBAL_SHAP_TOP{TOP_N}")
    # PDF (publication quality)
    plt.savefig(save_base + ".pdf",bbox_inches='tight',format='pdf')
    # PNG (quick viewing)
    plt.savefig(save_base + ".png",dpi=300,bbox_inches='tight',format='png')

    plt.close()
    
    print("✅ Global SHAP aggregation completed")
    # ==========================================
    # SHAP vs SEQUENCE LENGTH
    # ==========================================
    
    # Extract sequence length
    all_shap_df["dataset"] = all_shap_df["dataset"].astype(str)
    all_shap_df["seq_len"] = (all_shap_df["dataset"].str.replace(".csv", "", regex=False).astype(int))
    all_shap_df["seq_len"] = all_shap_df["seq_len"].astype(int)
    # Top 5 features globally
    
    plt.figure(figsize=(10, 6))

    all_lengths = sorted(all_shap_df["seq_len"].unique())
    
    for feat in top_features:
    
        subset = (all_shap_df[all_shap_df["feature"] == feat].sort_values("seq_len"))
    
        plt.plot(subset["seq_len"],subset["importance"],marker='o',label=feat)
    
    # Force ONLY integer ticks ()
    plt.xticks(all_lengths)
    # Optional extra safety (recommended for papers)
    plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    
    plt.xlabel("SPL Length")
    plt.ylabel("Mean |SHAP|")
    
    #plt.title(f"{model_name} Feature Importance vs Sequence Length")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    save_base = os.path.join("plots",model_name,f"{model_name}_SHAP_vs_SEQUENCE")
    # PDF (publication quality)
    plt.savefig(save_base + ".pdf",bbox_inches='tight',format='pdf')
    # PNG (quick viewing)
    plt.savefig(save_base + ".png",dpi=300,bbox_inches='tight',format='png')
    plt.close()
    print("✅ Global SHAP aggregation completed")

    plot_pred_vs_actual_subplots_pub(combined_results,model_name,save_dir=os.path.join("plots", model_name))
    # Save results
    results_df = pd.DataFrame(results)
    results_df.to_csv(f"{output_prefix}_summary.csv", index=False)

    all_cv_df = pd.concat(all_cv_results, ignore_index=True)
    all_cv_df.to_csv(f"{output_prefix}_cv_results.csv", index=False)
    best_metrics_df_all = pd.concat(best_metrics_list, ignore_index=True)
    best_metrics_df_all.to_csv(f"{output_prefix}_best_metrics.csv", index=False)
    print(f"\n✅ Summary saved to '{output_prefix}_summary.csv'")
    print(f"✅ Full CV results saved to '{output_prefix}_cv_results.csv'")
def plot_pred_vs_actual_subplots_pub(results_dict,model_name,save_dir="plots",marker_size=15):
    """
    Publication-ready Predicted vs Actual subplots
    with larger fonts and cleaner spacing.
    """

    import matplotlib.cm as cm
    from matplotlib.lines import Line2D

    os.makedirs(save_dir, exist_ok=True)

    num_datasets = len(results_dict)

    cols = 2
    rows = int(np.ceil(num_datasets / cols))

    # BIGGER FIGURE
    plt.figure(figsize=(6*cols, 5*rows))

    colors = cm.get_cmap('tab20', num_datasets)
    legend_labels = []

    for i, (dataset_name, (y_true, y_pred)) in enumerate(results_dict.items(), 1):

        ax = plt.subplot(rows, cols, i)

        n = 10
        yt = y_true[::n]
        yp = y_pred[::n]

        errors = np.abs(yt - yp)
        sc = ax.scatter(yt,yp,c=errors,cmap="coolwarm",s=marker_size,alpha=0.8)
        # SMALLER colorbar
        cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=10)
        cbar.set_label("Error", fontsize=11)
        # y=x reference line
        min_val = min(np.min(yt), np.min(yp))
        max_val = max(np.max(yt), np.max(yp))

        ax.plot([min_val, max_val],[min_val, max_val],'--',color='gray',lw=1.5)
        # Axis formatting
        ax.xaxis.set_major_formatter(lambda x, _: f"{int(x/1000)}k")
        ax.yaxis.set_major_formatter(lambda y, _: f"{int(y/1000)}k")
        # BIGGER LABELS
        ax.set_xlabel("Actual", fontsize=13)
        ax.set_ylabel("Predicted", fontsize=13)
        # BIGGER TICKS
        ax.tick_params(axis='both', labelsize=11)
        # Optional title
        ax.set_title(dataset_name, fontsize=13)
        ax.grid(True, alpha=0.3)
        legend_labels.append(f"k={i}")

    # Better spacing
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    # Common legend
    handles = [Line2D([0],[0],marker='o',color='w',markerfacecolor=colors(i),markersize=8)
        for i in range(num_datasets)]

    plt.figlegend(handles,legend_labels,loc='upper center',ncol=min(6, num_datasets),fontsize=12)
    save_base = os.path.join(save_dir,f"{model_name}_all_subplots_pub_fixed")
    # PDF (publication quality)
    plt.savefig(save_base + ".pdf",bbox_inches='tight',format='pdf')
    # PNG (quick viewing)
    plt.savefig(save_base + ".png",dpi=300,bbox_inches='tight',format='png')
    plt.show()


# LightGBM Model
Train , test and plot and save the models, plots, scalars, evaluations results in one folder for each model 

In [ ]:
from lightgbm import LGBMRegressor

LightGBM_model = LGBMRegressor(objective='regression', verbose=-1, random_state=42)

# Define hyperparameter grid
LightGBM_param_grid = {
    'model__n_estimators': [50, 200],
    'model__max_depth': [3, 10],  # -1 means no limit
    'model__num_leaves': [31, 127],  # default is 31
}
run_model_with_gridsearch(
    model=LightGBM_model,
    model_name='LightGBM',
    param_grid=LightGBM_param_grid,
    data_dict=df_filtered_dict,
    output_prefix='LightGBM_tuning')
zip_folder("saved_models", "saved_models.zip")
zip_folder("plots", "plots.zip")

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split


def export_small_test_subset(data_folder,output_folder="small_test_sets",original_test_size=0.2,subset_size=1000,random_state=42):

    os.makedirs(output_folder, exist_ok=True)

    for file in os.listdir(data_folder):

        if not file.endswith(".csv"):
            continue

        dataset_name = file.replace(".csv", "")

        path = os.path.join(data_folder, file)

        print(f"\n### Processing {dataset_name}")

        df = pd.read_csv(path)

        # preserve original row ids
        df = df.reset_index(drop=True)

       # df["orig_index"] = df.index

        # ==================================================
        # ORIGINAL TRAIN/TEST SPLIT (same as training code)
        # ==================================================

        train_df, test_df = train_test_split(df,test_size=original_test_size,random_state=random_state)

        # ==================================================
        # SMALL SUBSET FROM TEST SET
        # ==================================================
        if len(test_df) > subset_size:

            test_subset = test_df.sample(n=subset_size,random_state=random_state)

        else:

            test_subset = test_df.copy()
        # ==================================================
        # SAVE
        # ==================================================
        output_path = os.path.join(output_folder,f"{dataset_name}_TEST_SUBSET.csv")

        test_subset.to_csv(output_path, index=False)

        print(f"✅ Saved: {output_path}")
        print(f"   Original test size: {len(test_df)}")
        print(f"   Subset size: {len(test_subset)}")


# ======================================================
# RUN
# ======================================================

export_small_test_subset(data_folder=r"C:\Users\Dell\df_filtered_dictionary_90_97_eff_5_to_50_buffer",subset_size=1000,random_state=42)

In [ ]:
import os
import numpy as np
import pandas as pd

# =========================================================
# PATHS
# =========================================================

ENGINEERED_FOLDER = r"C:\Users\Dell\df_filtered_dictionary_90_97_eff_5_to_50_buffer"
OUTPUT_FOLDER = r"Packed_Original_Style"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
# =========================================================
# HELPERS
# =========================================================

def pack_matrices(row, max_m=20):
    matrices = []

    for i in range(1, max_m + 1):
        cols = [
            f"Matrix_{i}_Puu",
            f"Matrix_{i}_Pud",
            f"Matrix_{i}_Pdu",
            f"Matrix_{i}_Pdd"
        ]

        if all(c in row.index and not pd.isna(row[c]) for c in cols):
            mat = f"[{row[cols[0]]}, {row[cols[1]]}; {row[cols[2]]}, {row[cols[3]]}]"
            matrices.append(mat)
        else:
            matrices.append("None")
    return matrices

def pack_buffers(row):
    buffers = [row[c]
        for c in row.index
        if c.startswith("Buffer_")]
    return [b for b in buffers if not pd.isna(b)]

def rebuild_original_format(df):
    out = pd.DataFrame()
    # -----------------------------
    # BASIC FIELDS
    # -----------------------------
    out["Number of Machines"] = df["Number of Machines"]
    out["Buffers Capacities"] = df.apply(pack_buffers, axis=1)
    # -----------------------------
    # MATRICES
    # -----------------------------
    matrices = df.apply(pack_matrices, axis=1)
    max_m = len(matrices.iloc[0])

    for i in range(max_m):
        out[f"Matrix_{i+1}"] = matrices.apply(lambda x: x[i])

    # -----------------------------
    # OTHER COLUMNS (FILTERED)
    # -----------------------------
    drop_cols = {"Avg_NP1"}  # <-- remove this completely

    for col in df.columns:
        if col in out.columns:
            continue
        if col in drop_cols:
            continue
        if col.startswith("Buffer_") or col.startswith("Matrix_"):
            continue

        out[col] = df[col]

    return out


# =========================================================
# PROCESS ALL FILES
# =========================================================

for file in os.listdir(ENGINEERED_FOLDER):

    if not file.endswith(".csv"):
        continue

    path = os.path.join(ENGINEERED_FOLDER, file)
    print(f"\nProcessing: {file}")

    df = pd.read_csv(path)

    packed_df = rebuild_original_format(df)

    out_path = os.path.join(OUTPUT_FOLDER,file.replace(".csv", "_packed_original.csv"))

    packed_df.to_csv(out_path, index=False)

    print(f"Saved: {out_path}")

## Load the Model for predicting test cases from the Literature  

In [ ]:
import os
import pickle
import pandas as pd

import os
import pickle
import os
import time
import pickle
import pandas as pd

def load_lightgbm_model_by_key(
        key,
        model_root=r"D:\ML_SPL_Paper\LightGBm90to97and5to50buffer\saved_models\LightGBM"):

    model_filename = f"LightGBM_{key}_best.pkl"
    model_path = os.path.join(model_root, model_filename)

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found: {model_path}")

    with open(model_path, "rb") as f:
        model = pickle.load(f)

    return model


def add_predictions_to_dict(
        df_dict,
        target_col="Avg_NP1",
        sample_size=1000,
        timing_csv="inference_times.csv"):

    updated_dict = {}
    timing_results = []

    for key, df in df_dict.items():

        print(f"\n### Processing dataset with key = {key}")
        # =========================
        # Load model
        # =========================
        model = load_lightgbm_model_by_key(key)
        # =========================
        # Optional smaller subset
        # =========================
        if len(df) > sample_size:

            df = df.sample(n=sample_size,random_state=42).reset_index(drop=True)
        # =========================
        # Prepare features
        # =========================
        X = df.copy()

        if target_col in X.columns:
            X = X.drop(columns=[target_col])
        # =========================
        # Warm-up prediction
        # =========================
        _ = model.predict(X.iloc[:10])
        # =========================
        # Inference timing
        # =========================
        start = time.perf_counter()
        preds = model.predict(X)
        end = time.perf_counter()

        total_inference_time = end - start

        per_sample_time = (total_inference_time / len(X))
        # =========================
        # Save timing row
        # =========================
        timing_results.append({
            "dataset": key,
            "n_samples": len(X),
            "total_inference_time_sec": total_inference_time,
            "per_sample_inference_sec": per_sample_time})

        print(f"Samples: {len(X)}")
        print(f"Total inference time: {total_inference_time:.6f} sec")
        print(f"Per-sample inference: {per_sample_time:.9f} sec")
        # =========================
        # Save predictions
        # =========================
        df_with_preds = df.copy()
        df_with_preds[f"Predicted_{target_col}"] = preds
        updated_dict[key] = df_with_preds

        print(f"✔ Added prediction column to dataset {key}")
    # =========================
    # Save timing CSV
    # =========================
    timing_df = pd.DataFrame(timing_results)
    timing_df.to_csv(timing_csv,index=False)
    print(f"\n✔ Timing results saved to: {timing_csv}")
    return updated_dict

import os
import pandas as pd
def save_predicted_dataframes(df_dict, save_folder="predicted_results"):
    os.makedirs(save_folder, exist_ok=True)

    for key, df in df_dict.items():
        save_path = os.path.join(save_folder, f"{key}.csv")
        df.to_csv(save_path, index=False)
        print(f"✔ Saved predictions to {save_path}")

load_folder = r'C:\Users\Dell\small_test_sets'

rearranged_feature_selected_data_frame_dic = {
    k: v for k, v in sorted(
        {
            int(filename.split("_")[0]):
            pd.read_csv(os.path.join(load_folder, filename))
            for filename in os.listdir(load_folder)
            if filename.endswith('.csv')
        }.items())
}
df_with_predictions = add_predictions_to_dict(
    rearranged_feature_selected_data_frame_dic,
    sample_size=1000,
    timing_csv="LightGBM_inference_times.csv")
save_predicted_dataframes(
    df_with_predictions,
    save_folder="predicted_results"
)

# 6. Deep Learning for each length

In [ ]:
import zipfile
import os
import pandas as pd
import matplotlib.pyplot as plt # for plotting
import pickle
from sklearn.model_selection import train_test_split,cross_val_score, GridSearchCV,RandomizedSearchCV,ShuffleSplit  # Or StratifiedKFold
from tensorflow.keras.layers import GlobalAveragePooling2D, Input, Dense, RandomWidth, RandomHeight, RandomRotation, RandomFlip, RandomZoom, Conv2D, MaxPool2D, Flatten
import matplotlib.pyplot as plt # for plotting
from tensorflow.keras import Model
from keras.optimizers import Adam
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler,StandardScaler,PolynomialFeatures
from tensorflow.keras.models import Sequential, load_model


## Plotting function

In [ ]:
import pickle
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
#from scikeras.wrappers import KerasRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import os

def plot_pred_vs_actual_subplots_pub(results_dict,model_name,save_dir="plots",marker_size=15):
    """
    Publication-ready Predicted vs Actual subplots
    with larger fonts and cleaner spacing.
    """

    import matplotlib.cm as cm
    from matplotlib.lines import Line2D

    os.makedirs(save_dir, exist_ok=True)

    num_datasets = len(results_dict)

    cols = 4
    rows = int(np.ceil(num_datasets / cols))

    # BIGGER FIGURE
    plt.figure(figsize=(6*cols, 5*rows))

    colors = cm.get_cmap('tab20', num_datasets)
    legend_labels = []

    for i, (dataset_name, (y_true, y_pred)) in enumerate(results_dict.items(), 1):

        ax = plt.subplot(rows, cols, i)

        n = 10
        yt = y_true[::n]
        yp = y_pred[::n]

        errors = np.abs(yt - yp)
        sc = ax.scatter(yt,yp,c=errors,cmap="coolwarm",s=marker_size,alpha=0.8)
        # SMALLER colorbar
        cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=10)
        cbar.set_label("Error", fontsize=11)
        # y=x reference line
        min_val = min(np.min(yt), np.min(yp))
        max_val = max(np.max(yt), np.max(yp))

        ax.plot([min_val, max_val],[min_val, max_val],'--',color='gray',lw=1.5)
        # Axis formatting
        ax.xaxis.set_major_formatter(lambda x, _: f"{int(x/1000)}k")
        ax.yaxis.set_major_formatter(lambda y, _: f"{int(y/1000)}k")
        # BIGGER LABELS
        ax.set_xlabel("Actual", fontsize=13)
        ax.set_ylabel("Predicted", fontsize=13)
        # BIGGER TICKS
        ax.tick_params(axis='both', labelsize=11)
        # Optional title
        ax.set_title(dataset_name, fontsize=13)
        ax.grid(True, alpha=0.3)
        legend_labels.append(f"k={i}")

    # Better spacing
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    # Common legend
    handles = [Line2D([0],[0],marker='o',color='w',markerfacecolor=colors(i),markersize=8)
        for i in range(num_datasets)]

    plt.figlegend(handles,legend_labels,loc='upper center',ncol=min(6, num_datasets),fontsize=12)
    save_base = os.path.join(save_dir,f"{model_name}_all_subplots_pub_fixed")
    # PDF (publication quality)
    plt.savefig(save_base + ".pdf",bbox_inches='tight',format='pdf')
    # PNG (quick viewing)
    plt.savefig(save_base + ".png",dpi=300,bbox_inches='tight',format='png')
    plt.show()
def plot_training_results(y_true, y_pred, history_df, save_prefix=""):
    os.makedirs(save_prefix, exist_ok=True)  # Ensure folder exists

    # Sampled prediction plot
    n = 1000
    y_true_sampled = y_true[::n]
    y_pred_sampled = y_pred[::n]
    
    plt.figure(figsize=(8, 6))
    plt.plot(y_true_sampled, label="True")
    plt.plot(y_pred_sampled, label="Predicted", alpha=0.75)
   # plt.title("Test vs Predicted Output (Sampled)")
    plt.xlabel("Sample (every {}th point)".format(n))
    plt.ylabel("Average Throughput")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_prefix, "best_model_Actual_vs_Predicted_plot_sampled.png"))
    plt.close()

    # Scatter plot
    n = 500
    y_true_sampled = y_true[::n]
    y_pred_sampled = y_pred[::n]
    x_sampled = np.arange(len(y_true))[::n]

    plt.figure(figsize=(8, 6))
    plt.scatter(x_sampled, y_true_sampled, label="True", s=10)
    plt.scatter(x_sampled, y_pred_sampled, label="Predicted", s=10, alpha=0.7)
   # plt.title("Test vs Predicted Output (Scatter)")
    plt.xlabel("Observation Index")
    plt.ylabel("Average Throughput")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_prefix, "best_model_scatter_of_Actual_vs_Predicted_plot_scatter.png"))
    plt.close()

    # Loss and MAE curves
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history_df["loss"], label="Train Loss")
    if "val_loss" in history_df.columns:
        plt.plot(history_df["val_loss"], label="Val Loss")
    plt.title("Loss Over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history_df["mae"], label="Train MAE")
    if "val_mae" in history_df.columns:
        plt.plot(history_df["val_mae"], label="Val MAE")
    plt.title("MAE Over Epochs")
    plt.xlabel("Epoch")
    plt.ylabel("MAE")
    plt.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(save_prefix, "loss_mae_curves.png"))
    plt.close()
    # Residual plot
    residuals = y_true.flatten() - y_pred.flatten()
    plt.figure(figsize=(8, 5))
    plt.scatter(y_pred, residuals, alpha=0.6)
    plt.axhline(0, color='red', linestyle='--')
    plt.xlabel("Predicted")
    plt.ylabel("Residual (True - Predicted)")
    plt.title("Residual Plot")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(save_prefix, "residual_plot.png"))
    plt.close()

    return residuals


## Deep Learning Model trainging for each length 

In [ ]:
import os
import pickle
import time
import gc
import psutil
import joblib
import numpy as np
import pandas as pd
from itertools import product
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

def run_folds(train_idx, val_idx, X_train, y_train, params, model_builder, input_shape):
    model = model_builder(input_shape=input_shape, **params)

    early_stop = EarlyStopping(monitor="val_loss",patience=params.get("patience", 2),restore_best_weights=True,verbose=0)

    history = model.fit(X_train[train_idx], y_train[train_idx],validation_data=(X_train[val_idx], y_train[val_idx]),
                        epochs=params.get("epochs", 10),batch_size=params.get("batch_size", 32),verbose=0,callbacks=[early_stop])

    val_pred = model.predict(X_train[val_idx], verbose=0).reshape(-1, 1)
    y_val = y_train[val_idx].reshape(-1, 1)

    mse = mean_squared_error(y_val, val_pred)
    mae = mean_absolute_error(y_val, val_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_val, val_pred)
    rae = np.sum(np.abs(y_val - val_pred)) / np.sum(np.abs(y_val - np.mean(y_val)))
    rse = np.sum((y_val - val_pred) ** 2) / np.sum((y_val - np.mean(y_val)) ** 2)
   # MAPE = np.mean(np.abs(val_pred - y_val) / (y_val) ) # prevent div by zero

    fold_result = {
        "MSE": mse,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "RAE": rae,
        "RSE": rse,
       # "MAPE": MAPE,
        "Epochs": len(history.history["loss"])}

    K.clear_session()
    del model
    gc.collect()

    return fold_result, history
def generate_shap_plots_dl(model,X_train,X_test,feature_names,dataset_name,model_name,save_dir="plots"):

    shap_dir = os.path.join(save_dir, model_name, "shap")
    os.makedirs(shap_dir, exist_ok=True)

    background = X_train[:100]
    X_sample = X_test[:100]

    is_sequence = len(X_sample.shape) == 3

    try:

        explainer = shap.GradientExplainer(model, background)

        shap_values = explainer.shap_values(X_sample)

        if isinstance(shap_values, list):
            shap_values = shap_values[0]

        # =====================================
        # RESHAPE FOR SEQUENCE MODELS
        # =====================================
        feature_names = [f.replace("Matrix_", "Machine_")
            for f in feature_names]
        
        # =====================================
        # RESHAPE FOR SEQUENCE MODELS
        # =====================================
        if is_sequence:
            X_plot = X_sample.reshape(X_sample.shape[0], -1)
            shap_plot = shap_values.reshape(shap_values.shape[0], -1)
        
            # ONLY fallback if mismatch is unavoidable
            if len(feature_names) != X_plot.shape[1]:
                feature_names = [f"F{i}" for i in range(X_plot.shape[1])]
        else:
            X_plot = X_sample
            shap_plot = shap_values
        # =====================================
        # MEAN ABS SHAP
        # =====================================
        mean_abs_shap = np.abs(shap_plot).mean(axis=0)
        
        shap_df = pd.DataFrame({"feature": feature_names,
                                "importance": mean_abs_shap,
                                "dataset": dataset_name,
                                "model": model_name})
        # =====================================
        # SUMMARY PLOT
        # =====================================
        plt.figure(figsize=(12, 8))

        shap.summary_plot(shap_plot,X_plot,feature_names=feature_names,show=False)
        plt.tight_layout()
        save_base = os.path.join(shap_dir,f"{dataset_name}_summary"      )
        plt.savefig(save_base + ".pdf",bbox_inches='tight',format='pdf')
        plt.savefig(save_base + ".png",dpi=300,bbox_inches='tight',format='png')
        plt.close()
        # =====================================
        # BAR PLOT
        # =====================================
        plt.figure(figsize=(10, 7))
        shap.summary_plot(shap_plot,X_plot,feature_names=feature_names,plot_type="bar",show=False)
        plt.tight_layout()
        save_base = os.path.join(shap_dir,f"{dataset_name}_bar")
        plt.savefig(save_base + ".pdf",bbox_inches='tight',format='pdf')
        plt.savefig(save_base + ".png",dpi=300,bbox_inches='tight',format='png')
        plt.close()

        print(f"✅ DL SHAP plots saved for {dataset_name}")

        return shap_df

    except Exception as e:

        print(f"⚠️ SHAP failed for {dataset_name}")
        print(e)

        return None
def manual_grid_search_keras_parallel(df, target_column, model_builder, param_grid,
                                      test_size=0.2, random_state=42,
                                      model_type="dense", df_source_path=None,
                                      n_splits=5, n_jobs=-1):
    # Folder & naming
    dataset_name = os.path.splitext(os.path.basename(df_source_path))[0] if df_source_path else "Whole_SPL_Dataset"
    model_name = model_type.upper()
    folder_name = f"{model_name}_{dataset_name}"
    os.makedirs(folder_name, exist_ok=True)

    # Paths
    results_csv = os.path.join(folder_name, "manual_gridsearch_results.csv")
    best_csv = os.path.join(folder_name, "best_model_result.csv")
    model_save_path = os.path.join(folder_name, "best_model.pkl")
    scaler_X_path = os.path.join(folder_name, "scaler_X.pkl")
    scaler_y_path = os.path.join(folder_name, "scaler_y.pkl")
    summary_txt_path = os.path.join(folder_name, "best_model_summary.txt")

    X = np.nan_to_num(df.drop(columns=[target_column]).values, nan=0.0, posinf=0.0, neginf=0.0)
    y = np.nan_to_num(df[target_column].values.reshape(-1,1), nan=0.0, posinf=0.0, neginf=0.0)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    feature_names = df.drop(columns=[target_column]).columns.tolist()
    # Scale
    
    scaler_X = MinMaxScaler().fit(X_train)
    scaler_y = MinMaxScaler().fit(y_train)
    X_train_scaled = scaler_X.transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.transform(y_train).ravel()
    y_test_scaled = scaler_y.transform(y_test).ravel()
    # Reshape for 
    if model_type in ["RNN","Bi_RNN","Stacked_RNN","LSTM","Bi_LSTM","Stacked_LSTM","GRU","Seq2One"]:
        X_train_scaled = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
        X_test_scaled = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))
        input_shape = (1, X_train_scaled.shape[2])
    elif model_type in ["Conv1D","TCN","Transformer"]:
        X_train_scaled = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
        X_test_scaled = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))
        input_shape = (X_train_scaled.shape[1], 1)
    else:
        input_shape = X_train_scaled.shape[1]
    
    # Use X_train_scaled / y_train_scaled in KFold
    X_train, X_test = X_train_scaled, X_test_scaled
    y_train, y_test = y_train_scaled, y_test_scaled

    if isinstance(param_grid, list):
        combinations = param_grid
    else:
        keys = list(param_grid.keys())
        combinations = [dict(zip(keys, values)) for values in product(*param_grid.values())]

    best_score = float('inf')
    best_result = {}
    best_model = None
    best_history = None
    results = []
    combined_results = {}
    all_shap_results = []
    for combo_idx, params in enumerate(combinations):
        print(f"\n🔁 Testing combo {combo_idx+1}/{len(combinations)}: {params}")
        combo_start_time = time.time()
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

        # Parallel fold evaluation
        fold_results = Parallel(n_jobs=n_jobs)(
            delayed(run_folds)(train_idx, val_idx, X_train, y_train, params, model_builder, input_shape)
            for train_idx, val_idx in kf.split(X_train))

        # Aggregate fold results
        fold_metrics = {k: np.mean([fold_results[i][0][k] for i in range(len(fold_results))]) 
                        for k in fold_results[0][0].keys()}
        fold_epochs_avg = np.mean([fold_results[i][0]["Epochs"] for i in range(len(fold_results))])

        result = {**params, **fold_metrics, "Avg_Epochs": fold_epochs_avg, "Training_Time_sec": time.time()-combo_start_time}
        results.append(result)

        if fold_metrics["MSE"] < best_score:
            best_score = fold_metrics["MSE"]
            best_result = result
            # Save best model from first fold (for simplicity)
    best_train_start = time.time()
    best_model = model_builder(input_shape=input_shape,**{k: v for k, v in best_result.items() if k in param_grid})
    best_model.fit(X_train, y_train,epochs=int(best_result.get("epochs", 10)),
        batch_size=int(best_result.get("batch_size", 32)),verbose=0)
    best_training_time = time.time() - best_train_start
    # ===== SHAP =====
    shap_df = generate_shap_plots_dl(model=best_model,X_train=X_train,X_test=X_test,feature_names=feature_names,
                                     dataset_name=dataset_name,model_name=model_name)
    if shap_df is not None:
        all_shap_results.append(shap_df)
    # Test evaluation
    #y_test_pred_scaled = best_model.predict(X_test).reshape(-1,1)
    # ==============================
    # INFERENCE TIME MEASUREMENT
    # ==============================
    # warm-up (removes cold start bias)
    _ = best_model.predict(X_test[:10], verbose=0)
    
    n_test = len(X_test)
    
    start = time.perf_counter()
    
    y_test_pred_scaled = best_model.predict(X_test, verbose=0).reshape(-1, 1)
    
    end = time.perf_counter()
    
    inference_time_total = end - start
    inference_time_per_sample = inference_time_total / n_test
    y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled)
    y_test_orig = scaler_y.inverse_transform(y_test.reshape(-1,1))
    all_results_dict[dataset_name] = (y_test_orig, y_test_pred)
    mse = mean_squared_error(y_test_orig, y_test_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_orig, y_test_pred)
    r2 = r2_score(y_test_orig, y_test_pred)
    rae = np.sum(np.abs(y_test_orig-y_test_pred))/np.sum(np.abs(y_test_orig-np.mean(y_test_orig)))
    rse = np.sum((y_test_orig-y_test_pred)**2)/np.sum((y_test_orig-np.mean(y_test_orig))**2)
    MAPE = np.mean(np.abs(y_test_pred - y_test_orig) / (y_test_orig))  # prevent div by zero
    
    best_result.update({"Test_MSE": mse, "Test_RMSE": rmse, "Test_MAE": mae, "Test_R2": r2, "Test_RAE": rae,
                        "Test_RSE": rse,"Test_MAPE":MAPE,"best_training_time":best_training_time, "n_test_samples": n_test,
                        "inference_time_total_sec": inference_time_total,"inference_time_per_sample_sec": inference_time_per_sample})
    #plot_pred_vs_actual_subplots_pub(combined_results,model_name,save_dir=os.path.join("plots", model_name))
    # Save
    pd.DataFrame(results).to_csv(results_csv, index=False)
    pd.DataFrame([best_result]).to_csv(best_csv, index=False)
    joblib.dump(scaler_X, scaler_X_path)
    joblib.dump(scaler_y, scaler_y_path)
    with open(model_save_path, "wb") as f:
        pickle.dump(best_model, f)
    with open(summary_txt_path, "w", encoding="utf-8") as f:
        best_model.summary(print_fn=lambda x: f.write(x + "\n"))
    # ==========================================
    # GLOBAL SHAP AGGREGATION
    # ==========================================
    if len(all_shap_results) > 0:

        all_shap_df = pd.concat(all_shap_results,ignore_index=True)
        all_shap_df["feature"] = (all_shap_df["feature"].str.replace("Matrix_", "Machine_", regex=False))
        all_shap_df["dataset"] = (all_shap_df["dataset"].astype(str))
    
        all_shap_df["seq_len"] = (all_shap_df["dataset"].str.replace(".csv", "", regex=False).astype(int))
        # ======================================
        # GLOBAL FEATURE IMPORTANCE
        # ======================================
        global_shap = (all_shap_df.groupby("feature")["importance"].mean().sort_values(ascending=False).reset_index())
        top_features = (global_shap["feature"].head(5).tolist())
        TOP_N = 10
        global_shap_top = global_shap.head(TOP_N)
    
        plt.figure(figsize=(8, 6))
    
        plt.barh(global_shap_top["feature"],global_shap_top["importance"])
    
        plt.gca().invert_yaxis()
        plt.xlabel("Mean |SHAP value|")
        plt.ylabel("Feature")
        plt.tight_layout()
        save_base = os.path.join("plots",model_name,f"{model_name}_GLOBAL_SHAP_TOP{TOP_N}")
        plt.savefig(save_base + ".pdf",bbox_inches='tight',format='pdf')
        plt.savefig(save_base + ".png",dpi=300,bbox_inches='tight',format='png')
        plt.close()
        # ======================================
        # SHAP VS SEQUENCE LENGTH
        # ======================================
        plt.figure(figsize=(10, 6))

        all_lengths = sorted(all_shap_df["seq_len"].unique())
        
        for feat in top_features:
            subset = (all_shap_df[all_shap_df["feature"] == feat].sort_values("seq_len"))
            plt.plot(subset["seq_len"],subset["importance"],marker='o',label=feat)
        
        # Force integer-only x-axis ticks
        plt.xticks(all_lengths)
        # extra safety (recommended)
        plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        
        plt.xlabel("SPL Length")
        plt.ylabel("Mean |SHAP|")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
    
        save_base = os.path.join("plots",model_name,f"{model_name}_SHAP_vs_SPL Length")
        plt.savefig(save_base + ".pdf",bbox_inches='tight',format='pdf')
    
        plt.savefig(save_base + ".png", dpi=300,bbox_inches='tight',format='png')
    
        plt.close()
    
        print("✅ DL Global SHAP aggregation completed")
    print("\n✅ Best Result:")
    print(best_result)
    return pd.DataFrame(results), pd.DataFrame([best_result])


# Training model for each length of SPL alone
### Loading the data

In [ ]:
# --- Load CSVs ---
load_folder = r'D:\delete later\df_filtered_dictionary_90_97_eff_5_to_50_buffer'
df_filtered_dict = {
    k: v for k, v in sorted(
        {
            int(os.path.splitext(filename)[0]): pd.read_csv(os.path.join(load_folder, filename))
            for filename in os.listdir(load_folder)
            if filename.endswith('.csv')
        }.items())
}

## LSTM for each SPL Length

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Masking
from tensorflow.keras.optimizers import Adam
import shap
def build_lstm_model(input_shape, lstm_units=32, dense_units=32, epochs=10,batch_size=16,dropout_rate=0.3, learning_rate=1e-3):
    model = Sequential([Masking(mask_value=0.0, input_shape=input_shape),  # <-- Add this line for masking padded zeros
                        LSTM(lstm_units, activation='tanh', input_shape=input_shape, return_sequences=False),
                        BatchNormalization(),Dropout(dropout_rate),Dense(dense_units, activation='relu'),
                        Dropout(dropout_rate),Dense(1)])
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse', metrics=['mae'])
    return model
param_grid_lstm = {'lstm_units': [32,64],'dense_units': [16,32],
                   'dropout_rate': [0.2],'learning_rate': [1e-3],
                   'batch_size': [16],'epochs': [15]} 
all_results_dict = {}
for file_id, df in df_filtered_dict.items():
    # Construct file path
    filename = f"{file_id}.csv"
    df_path = os.path.join(load_folder, filename)
    # Call the function
    results,best_result= manual_grid_search_keras_parallel(
        df=df,
        target_column="Avg_NP1",
        model_builder=build_lstm_model,
        param_grid=param_grid_lstm,
        model_type="LSTM",  # or any other modela
        df_source_path=df_path)
plot_pred_vs_actual_subplots_pub(
    all_results_dict,
    model_name="lstm",
    save_dir=os.path.join("plots", "lstm"))    

# Load data for prediction of test cases

In [ ]:
# --- Load CSVs ---
load_folder = r'C:\Users\Dell\LSTM'
df_validated_dict = {
    k: v for k, v in sorted(
        {
            int(os.path.splitext(filename)[0]): pd.read_csv(os.path.join(load_folder, filename))
            for filename in os.listdir(load_folder)
            if filename.endswith('.csv')
        }.items()
    )
}

# Load the trained model for test cases for validation agaist the Literature   

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import pickle
import joblib

# =========================================================
# MODEL LOADING
# =========================================================

def load_lstm_model(model_folder):
    model_path = os.path.join(model_folder, "best_model.pkl")
    scaler_X_path = os.path.join(model_folder, "scaler_X.pkl")
    scaler_y_path = os.path.join(model_folder, "scaler_y.pkl")

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found: {model_path}")

    with open(model_path, "rb") as f:
        model = pickle.load(f)

    scaler_X = joblib.load(scaler_X_path)
    scaler_y = joblib.load(scaler_y_path)

    return model, scaler_X, scaler_y


# =========================================================
# PREDICTION FUNCTION
# =========================================================

def predict_with_lstm(df, target_col, model, scaler_X, scaler_y):
    X = df.drop(columns=[target_col]).values

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    X_scaled = scaler_X.transform(X)

    # LSTM expects [samples, timesteps, features]
    X_scaled = X_scaled.reshape((X_scaled.shape[0], 1, X_scaled.shape[1]))

    y_pred_scaled = model.predict(X_scaled, verbose=0)
    y_pred = scaler_y.inverse_transform(y_pred_scaled)

    return y_pred.ravel()


# =========================================================
# MAIN PIPELINE (WITH TIMING)
# =========================================================

def add_lstm_predictions_to_dict(
        df_dict,
        model_root_folder=".",
        target_col="Avg_NP1",
        sample_size=1000,
        timing_csv="lstm_inference_times.csv"):

    updated_dict = {}
    timing_results = []

    for file_id, df in df_dict.items():

        print(f"\n### Processing dataset {file_id}")

        # -------------------------
        # locate model folder
        # -------------------------
        model_folder = os.path.join(model_root_folder, f"LSTM_{file_id}")

        print(f"Loading model: {model_folder}")

        model, scaler_X, scaler_y = load_lstm_model(model_folder)

        # -------------------------
        # optional sampling
        # -------------------------
        if len(df) > sample_size:
            df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)

        # -------------------------
        # prepare input
        # -------------------------
        X = df.copy()

        if target_col in X.columns:
            X = X.drop(columns=[target_col])

        # -------------------------
        # warm-up (important for LSTM GPU/TF stability)
        # -------------------------
        X_warm = X.iloc[:10].copy()

        X_warm = np.nan_to_num(X_warm.values, nan=0.0, posinf=0.0, neginf=0.0)
        X_warm = scaler_X.transform(X_warm)
        
        X_warm = X_warm.reshape((X_warm.shape[0], 1, X_warm.shape[1]))
        
        _ = model.predict(X_warm, verbose=0)
        # -------------------------
        # inference timing
        # -------------------------
        start = time.perf_counter()

        y_pred = predict_with_lstm(df, target_col, model, scaler_X, scaler_y)

        end = time.perf_counter()

        total_time = end - start
        per_sample_time = total_time / len(df)

        # -------------------------
        # store timing
        # -------------------------
        timing_results.append({
            "dataset": file_id,
            "n_samples": len(df),
            "total_inference_time_sec": total_time,
            "per_sample_inference_sec": per_sample_time
        })

        print(f"Samples: {len(df)}")
        print(f"Total time: {total_time:.6f} sec")
        print(f"Per sample: {per_sample_time:.9f} sec")

        # -------------------------
        # save predictions
        # -------------------------
        df_new = df.copy()
        df_new["Predicted_Avg_NP1"] = y_pred

        updated_dict[file_id] = df_new

        print(f"✔ Predictions added for {file_id}")

    # -------------------------
    # save timing CSV
    # -------------------------
    timing_df = pd.DataFrame(timing_results)
    timing_df.to_csv(timing_csv, index=False)

    print(f"\n✔ Timing saved to: {timing_csv}")

    return updated_dict


# =========================================================
# SAVE RESULTS
# =========================================================

def save_predicted_frames(df_dict, save_folder="lstm_predictions"):
    os.makedirs(save_folder, exist_ok=True)

    for file_id, df in df_dict.items():
        save_path = os.path.join(save_folder, f"{file_id}.csv")
        df.to_csv(save_path, index=False)
        print(f"✔ Saved: {save_path}")
load_folder = r"C:\Users\Dell\small_test_sets"

df_dict = {
    int(filename.split("_")[0]): pd.read_csv(os.path.join(load_folder, filename))
    for filename in os.listdir(load_folder)
    if filename.endswith(".csv")
}

df_dict = dict(sorted(df_dict.items()))

df_with_predictions = add_lstm_predictions_to_dict(
    df_dict,
    model_root_folder=r"D:\ML_SPL_Paper\LSTM\New folder",
    target_col="Avg_NP1",
    sample_size=1000,
    timing_csv="LSTM_inference_times.csv"
)

save_predicted_frames(
    df_with_predictions,
    save_folder="lstm_predictions"
)